# Energy Infrastructure Supply Chain Analysis

This notebook applies the **Critical Supply Chain Resilience AI** prototype to a synthetic **energy infrastructure equipment** supply network (grid transformers, inverters, and deployment hub).

**Use case:** Evaluate supplier dependencies and disruption impact for critical energy infrastructure components.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import plotly.io as pio
pio.renderers.default = "notebook"

from scripts.prepare_energy_data import prepare_energy_data
from src.pipeline import run_prototype_pipeline
from src.utils.config import ENERGY_DATA_DIR
from src.utils.graph_builder import build_supply_network
from src.visualization.plotly_charts import (
    demand_forecast_chart,
    mitigation_chart,
    network_graph_plotly,
    simulation_chart,
    supplier_risk_chart,
)

In [ ]:
prepare_energy_data()
report = run_prototype_pipeline(data_dir=ENERGY_DATA_DIR)
graph = build_supply_network(ENERGY_DATA_DIR)
print("Energy network:", report.network_summary)

## Network Topology

Interactive map of suppliers, assembly plants, and the national grid deployment hub.

In [ ]:
network_graph_plotly(graph).show()

## Resilience Metrics

In [ ]:
print(f"Network Risk Index: {report.resilience.network_risk_index}")
print(f"Supplier Dependency: {report.resilience.supplier_dependency_score}")
print(f"Estimated Recovery Days: {report.resilience.estimated_recovery_days}")
report.resilience.node_risks.head(8)

## Supplier Disruption Risk

In [ ]:
supplier_risk_chart(report).show()
report.disruption_predictions

## Disruption Simulations

Scenarios include permanent magnet supplier outage, grid control software outage, and Gulf Coast plant shutdown.

In [ ]:
simulation_chart(report).show()
for result in report.simulations:
    print(
        f"{result.scenario.description}: service level {result.service_level:.1%}, "
        f"recovery ~{result.recovery_days_estimate} days"
    )

## Mitigation Recommendations

In [ ]:
mitigation_chart(report).show()
for item in report.mitigations:
    print(f"{item.priority}. {item.action} for {item.target_node} ({item.material})")
    print(f"   {item.rationale}")

## Demand Forecast

In [ ]:
demand_forecast_chart(report).show()
report.demand_forecast

## Single-Source Dependencies

These are priority targets for supplier diversification in energy infrastructure manufacturing.

In [ ]:
report.resilience.single_source_exposure